# Biomarker Brief

A 100% local pipeline that turns a stack of lab PDFs into a plain-English
briefing you can take to your doctor.

> **Not medical advice. Educational only. Bring this to your doctor; do not
> act on it. Reference ranges vary by lab, age, sex, and clinical context.
> Your labs never leave this machine.**

**How it works** — eight cells, run them top to bottom:

1. **Setup** — install Python deps, verify Ollama is running, pick a model.
2. **Ingest** — read every PDF / CSV in `./data/sample_labs/`.
3. **Parse** — extract `{marker, value, unit, ref_range}` (regex first, LLM fallback).
4. **Normalize** — map vendor-specific names to canonical names.
5. **Flag** — mark out-of-range values and classify severity.
6. **Trend** — plot any marker measured ≥2 times; flag worsening trends even if in-range.
7. **Brief** — Ollama writes per-marker explanations + an executive summary + questions for your doctor.
8. **Export** — `brief.md`, `trends.png`, `flagged_markers.csv` → `./output/`.

**Privacy.** Every step runs on this machine. No outbound network calls to any
LLM provider. The only external dependency is the Python package index during
the initial `pip install` in cell 1.


## Cell 1 — Setup

Install Python deps, verify Ollama, pick a model.

In [ ]:
# --- Cell 1: Setup -------------------------------------------------------
import subprocess, sys, importlib, json, urllib.request, urllib.error

REQUIRED = ["pdfplumber", "pypdf", "pandas", "matplotlib", "requests"]

def _ensure(pkg: str) -> None:
    name = pkg.split("==")[0]
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

for p in REQUIRED:
    _ensure(p)

import requests  # noqa: E402  (after install)

# --- Verify Ollama is running locally ------------------------------------
OLLAMA_HOST = "http://localhost:11434"

def ollama_running() -> bool:
    try:
        r = urllib.request.urlopen(f"{OLLAMA_HOST}/api/tags", timeout=2)
        return r.status == 200
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        return False

if not ollama_running():
    print(
        "Ollama is not reachable at http://localhost:11434.\n"
        "  Mac/Win: install from https://ollama.com and launch the app.\n"
        "  Linux:   `curl -fsSL https://ollama.com/install.sh | sh` then `ollama serve`\n"
        "Then come back and re-run this cell."
    )
else:
    tags = json.loads(urllib.request.urlopen(f"{OLLAMA_HOST}/api/tags").read())
    installed = [m["name"] for m in tags.get("models", [])]
    print("Ollama is up. Installed models:", installed or "(none yet)")

# --- Pick a model --------------------------------------------------------
# Default is small + fast. Suggested upgrades for better medical reasoning:
#   ollama pull llama3.1:8b       (~5 GB, much better summaries)
#   ollama pull qwen2.5:14b       (~9 GB, slower but stronger)
#   ollama pull meditron:7b       (medical-tuned community model)
OLLAMA_MODEL = "llama3.2"

print(f"\nUsing model: {OLLAMA_MODEL}")
print("If you have not pulled it yet, run in a terminal:")
print(f"    ollama pull {OLLAMA_MODEL}")


## Cell 2 — Ingest

Read every PDF and CSV under `./data/sample_labs/` and extract a draw date.

We try, in order:
1. A date in the filename (e.g. `labs_2024-09-18_labcorp.pdf`).
2. A date pattern inside the document body.
3. The file's modification time, as a last resort.


In [ ]:
# --- Cell 2: Ingest ------------------------------------------------------
from pathlib import Path
from datetime import datetime
import re
import pdfplumber
import pandas as pd

DATA_DIR = Path("../data/sample_labs")
OUTPUT_DIR = Path("../output"); OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

DATE_PATTERNS = [
    re.compile(r"\b(20\d{2})[-/.](\d{1,2})[-/.](\d{1,2})\b"),   # 2024-09-18
    re.compile(r"\b(\d{1,2})[-/.](\d{1,2})[-/.](20\d{2})\b"),    # 09/18/2024
]

def _parse_date(s: str) -> datetime | None:
    for pat in DATE_PATTERNS:
        m = pat.search(s)
        if not m:
            continue
        g = m.groups()
        try:
            if len(g[0]) == 4:
                y, mo, d = int(g[0]), int(g[1]), int(g[2])
            else:
                mo, d, y = int(g[0]), int(g[1]), int(g[2])
            return datetime(y, mo, d)
        except ValueError:
            continue
    return None

def _read_pdf_text(path: Path) -> str:
    chunks = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            chunks.append(page.extract_text() or "")
    return "\n".join(chunks)

def ingest(data_dir: Path) -> list[dict]:
    docs = []
    files = sorted([p for p in data_dir.iterdir() if p.suffix.lower() in {".pdf", ".csv"}])
    for path in files:
        if path.suffix.lower() == ".pdf":
            text = _read_pdf_text(path)
            kind = "pdf"
        else:
            text = path.read_text(encoding="utf-8", errors="replace")
            kind = "csv"

        draw_date = (
            _parse_date(path.name)
            or _parse_date(text[:2000])
            or datetime.fromtimestamp(path.stat().st_mtime)
        )

        docs.append({
            "path": str(path),
            "filename": path.name,
            "kind": kind,
            "draw_date": draw_date.date().isoformat(),
            "text": text,
        })
    return docs

DOCS = ingest(DATA_DIR)
print(f"Ingested {len(DOCS)} document(s):")
for d in DOCS:
    print(f"  • {d['draw_date']}  {d['filename']}  ({d['kind']}, {len(d['text'])} chars)")


## Cell 3 — Parse

Pull `{marker, value, unit, ref_range}` rows out of each document.

- **Regex first.** Most vendor reports lay out one result per line as
  `Test Name   Value   Unit   Reference Range   Flag`.  We match that pattern,
  with tolerance for noisy whitespace.
- **CSV passthrough.** If the document is a CSV (e.g. a portal export),
  parse it as a table directly.
- **LLM fallback.** Anything that didn't yield rows goes to Ollama, which
  is asked to return strict JSON.  No network — runs against your local
  Ollama server.


In [ ]:
# --- Cell 3: Parse -------------------------------------------------------
import io, json, re, requests
import pandas as pd

UNITS = (
    r"mg/dL|mg/L|g/dL|ng/mL|ng/dL|pg/mL|ug/dL|ug/L|nmol/L|mmol/L|umol/L|"
    r"uIU/mL|mIU/L|IU/mL|U/L|%|fL|pg|10\^3/uL|10\^6/uL|"
    r"mL/min/1\.73m2|ratio"
)

LINE_RE = re.compile(
    r"^(?P<name>[A-Za-z][A-Za-z0-9 ()/,.\-]{2,55}?)\s+"
    r"(?P<value>[<>]?\s*\d+\.?\d*)\s+"
    rf"(?P<unit>{UNITS})\s+"
    r"(?P<ref>[<>]?\s*[\d.\-=\s/]+?)"
    r"(?:\s+(?P<flag>H(?:IGH)?|L(?:OW)?|HH|LL|\*))?\s*$",
    re.IGNORECASE,
)


def parse_text_regex(text: str) -> list[dict]:
    """Line-by-line regex parse of a vendor PDF dump."""
    rows = []
    for raw in text.splitlines():
        line = " ".join(raw.split())
        m = LINE_RE.match(line)
        if not m:
            continue
        rows.append({
            "marker_raw": m.group("name").strip(),
            "value": m.group("value").replace(" ", ""),
            "unit": m.group("unit"),
            "ref_range": m.group("ref").strip(),
            "flag": (m.group("flag") or "").upper(),
        })
    return rows


def parse_csv(text: str) -> list[dict]:
    """Parse a generic CSV export. Columns are best-effort matched."""
    df = pd.read_csv(io.StringIO(text))
    cols = {c.lower().strip(): c for c in df.columns}
    def pick(*names):
        for n in names:
            if n in cols:
                return cols[n]
        return None
    name_c = pick("marker", "test", "test name", "analyte")
    val_c  = pick("value", "result")
    unit_c = pick("unit", "units")
    ref_c  = pick("reference_range", "reference range", "range", "ref")
    flag_c = pick("flag", "abnormal")
    date_c = pick("draw_date", "collected", "date")
    if not (name_c and val_c):
        return []
    out = []
    for _, r in df.iterrows():
        out.append({
            "marker_raw": str(r[name_c]).strip(),
            "value": str(r[val_c]).strip(),
            "unit": str(r[unit_c]).strip() if unit_c else "",
            "ref_range": str(r[ref_c]).strip() if ref_c else "",
            "flag": (str(r[flag_c]).strip().upper() if flag_c else ""),
            "_csv_draw_date": str(r[date_c]).strip() if date_c else None,
        })
    return out


def parse_with_llm(text: str, model: str = None) -> list[dict]:
    """Ask the local Ollama model to return strict JSON rows. Used as fallback."""
    model = model or OLLAMA_MODEL
    snippet = text[:6000]
    prompt = (
        "Extract every laboratory test result from the report below. Return "
        "ONLY a JSON array (no prose, no markdown fences). Each element must "
        'have keys: "marker_raw", "value", "unit", "ref_range", "flag". '
        "If a field is unknown, use an empty string. Skip header lines, "
        "comments, and any non-result text.\n\n"
        f"REPORT:\n{snippet}\n\nJSON:"
    )
    try:
        r = requests.post(
            f"{OLLAMA_HOST}/api/generate",
            json={"model": model, "prompt": prompt, "stream": False,
                  "options": {"temperature": 0}},
            timeout=120,
        )
        raw = r.json().get("response", "")
        m = re.search(r"\[.*\]", raw, re.S)
        if not m:
            return []
        return json.loads(m.group(0))
    except Exception as e:
        print(f"  (LLM fallback failed: {e})")
        return []


def parse_doc(doc: dict) -> list[dict]:
    if doc["kind"] == "csv":
        rows = parse_csv(doc["text"])
    else:
        rows = parse_text_regex(doc["text"])
        if len(rows) < 3:
            print(f"  regex got only {len(rows)} rows from {doc['filename']}; trying LLM ...")
            rows = parse_with_llm(doc["text"]) or rows
    for r in rows:
        r["source_file"] = doc["filename"]
        r["draw_date"] = r.pop("_csv_draw_date", None) or doc["draw_date"]
    return rows


PARSED: list[dict] = []
for d in DOCS:
    rows = parse_doc(d)
    PARSED.extend(rows)
    print(f"  • {d['filename']}: parsed {len(rows)} rows")

print(f"\nTotal raw rows: {len(PARSED)}")
pd.DataFrame(PARSED).head(10)


## Cell 4 — Normalize

Vendors disagree on naming. We map every observed string to a *canonical*
name using `data/biomarker_reference.json`, with case-insensitive matching
on the `also_known_as` list. Unrecognized markers are kept but flagged
`unknown=True` so you can see what we missed.


In [ ]:
# --- Cell 4: Normalize ---------------------------------------------------
import json
from pathlib import Path

REF_PATH = Path("../data/biomarker_reference.json")
REFERENCE = json.loads(REF_PATH.read_text())
MARKERS = REFERENCE["markers"]

# Build a one-shot alias lookup, lowercase + stripped.
ALIAS_MAP: dict[str, dict] = {}
for m in MARKERS:
    names = [m["canonical_name"], *m["also_known_as"]]
    for n in names:
        ALIAS_MAP[n.lower().strip()] = m


def _normalize_name(raw: str) -> str:
    s = raw.lower().strip()
    s = re.sub(r"\s*\([^)]*\)", "", s)        # drop parentheticals
    s = re.sub(r",\s*(serum|plasma|calculated|calc|direct|total)\s*$", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip(" ,.:;-")


def normalize_row(row: dict) -> dict:
    key = _normalize_name(row["marker_raw"])
    ref = ALIAS_MAP.get(key)
    if ref is None:
        # try a looser contains-match against canonical names
        for cand_key, cand in ALIAS_MAP.items():
            if cand_key in key or key in cand_key:
                ref = cand
                break
    out = dict(row)
    if ref:
        out["marker"] = ref["canonical_name"]
        out["category"] = ref["category"]
        out["unknown"] = False
    else:
        out["marker"] = row["marker_raw"]
        out["category"] = "unknown"
        out["unknown"] = True
    # coerce numeric value
    try:
        out["value_num"] = float(re.sub(r"[<>\s]", "", str(row["value"])))
    except ValueError:
        out["value_num"] = None
    return out


NORMALIZED = [normalize_row(r) for r in PARSED]
df = pd.DataFrame(NORMALIZED)
print(f"Normalized {len(df)} rows; {df['unknown'].sum()} unknown markers")
if df["unknown"].any():
    print("Unknown markers:", sorted(df.loc[df["unknown"], "marker_raw"].unique()))
df[["draw_date", "marker", "value", "unit", "category", "unknown"]].head(15)


## Cell 5 — Flag

For every recognized marker we compare the measured value to the reference
ranges in `biomarker_reference.json`:

- `in_range` — within `optimal_range`.
- `mild_out` — outside optimal but still inside `clinical_range` (your lab
  would not put a flag on this — but it's worth noticing).
- `significant_out` — outside `clinical_range`. These are the ones your lab
  flags `H` / `L`.

We trust the reference JSON here, not the vendor's printed range, so the
output is consistent across labs.


In [ ]:
# --- Cell 5: Flag --------------------------------------------------------
def classify(value: float | None, ref: dict) -> tuple[str, str]:
    """Return (status, direction). status in {in_range, mild_out, significant_out}."""
    if value is None:
        return "unparseable", ""
    o_lo, o_hi = ref["optimal_range"]
    c_lo, c_hi = ref["clinical_range"]
    if c_lo <= value <= c_hi and o_lo <= value <= o_hi:
        return "in_range", ""
    direction = "high" if value > o_hi else "low"
    if c_lo <= value <= c_hi:
        return "mild_out", direction
    return "significant_out", direction


by_marker_ref = {m["canonical_name"]: m for m in MARKERS}

def annotate(row: dict) -> dict:
    out = dict(row)
    if row["unknown"] or row["value_num"] is None:
        out["status"] = "unparseable" if row["value_num"] is None else "unknown_marker"
        out["direction"] = ""
        return out
    ref = by_marker_ref[row["marker"]]
    status, direction = classify(row["value_num"], ref)
    out["status"] = status
    out["direction"] = direction
    out["optimal_range"] = ref["optimal_range"]
    out["clinical_range"] = ref["clinical_range"]
    return out


FLAGGED = [annotate(r) for r in NORMALIZED]
fdf = pd.DataFrame(FLAGGED)

summary = (
    fdf.groupby("status").size().rename("count").reset_index()
    if not fdf.empty else pd.DataFrame()
)
print("Status summary:")
print(summary.to_string(index=False) if not summary.empty else "(no rows)")

print("\nOut-of-range (any severity), most recent draw:")
latest = fdf["draw_date"].max() if not fdf.empty else None
flagged_latest = fdf[
    (fdf["draw_date"] == latest)
    & (fdf["status"].isin(["mild_out", "significant_out"]))
]
flagged_latest[["marker", "value", "unit", "direction", "status", "optimal_range"]]


## Cell 6 — Trend

For any marker measured on ≥2 dates, plot the trajectory with the optimal
range shaded. We also compute a delta and a simple *trend status*:

- `worsening` — value is moving *away from* the optimal midpoint over time
  (even if still inside the range).
- `improving` — value is moving *toward* the optimal midpoint.
- `stable` — small change.

A worsening-but-still-in-range trend is exactly the kind of thing single
snapshots miss.


In [ ]:
# --- Cell 6: Trend -------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

def _trend_status(values: list[float], ref: dict) -> tuple[str, float]:
    if len(values) < 2:
        return "single_point", 0.0
    o_lo, o_hi = ref["optimal_range"]
    mid = (o_lo + o_hi) / 2.0
    dist_first = abs(values[0] - mid)
    dist_last = abs(values[-1] - mid)
    delta = values[-1] - values[0]
    rel_change = (dist_last - dist_first) / max(abs(mid), 1e-9)
    if rel_change > 0.10:
        return "worsening", delta
    if rel_change < -0.10:
        return "improving", delta
    return "stable", delta


# Group by canonical marker, sorted by date.
TRENDS: dict[str, list[dict]] = {}
for r in FLAGGED:
    if r["unknown"] or r["value_num"] is None:
        continue
    TRENDS.setdefault(r["marker"], []).append(r)

trend_rows = []
plottable = {m: sorted(rows, key=lambda x: x["draw_date"])
             for m, rows in TRENDS.items() if len(rows) >= 2}

print(f"{len(plottable)} marker(s) measured ≥2 times.\n")

# Plot grid
if plottable:
    n = len(plottable)
    cols = min(3, n)
    rows_n = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 3.2 * rows_n), squeeze=False)
    for idx, (marker, rows) in enumerate(plottable.items()):
        ax = axes[idx // cols][idx % cols]
        ref = by_marker_ref[marker]
        dates = [datetime.fromisoformat(r["draw_date"]) for r in rows]
        vals = [r["value_num"] for r in rows]
        ax.plot(dates, vals, marker="o", linewidth=2, color="#2d5040")
        ax.axhspan(*ref["optimal_range"], alpha=0.15, color="#5a8a5e", label="optimal")
        ax.set_title(f"{marker} ({ref['unit']})", fontsize=10)
        ax.tick_params(axis="x", rotation=30, labelsize=8)
        ax.tick_params(axis="y", labelsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        status, delta = _trend_status(vals, ref)
        ax.annotate(
            f"Δ {delta:+.2f} • {status}",
            xy=(0.02, 0.95), xycoords="axes fraction",
            fontsize=8, va="top",
            color={"worsening": "#a01010", "improving": "#2d5040"}.get(status, "#555"),
        )
        trend_rows.append({
            "marker": marker,
            "n_points": len(vals),
            "first_value": vals[0],
            "last_value": vals[-1],
            "delta": delta,
            "trend": status,
        })
    # blank unused axes
    for j in range(len(plottable), rows_n * cols):
        axes[j // cols][j % cols].axis("off")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "trends.png", dpi=120, bbox_inches="tight")
    print(f"saved {OUTPUT_DIR / 'trends.png'}")
    plt.show()

trend_df = pd.DataFrame(trend_rows).sort_values("trend") if trend_rows else pd.DataFrame()
trend_df


## Cell 7 — Brief

We hand the structured results to Ollama and ask for three things:

1. A short **executive summary** of what changed between the earliest and
   latest draws.
2. **Per-marker explanations** for every flagged or trending marker — what
   it measures, what high/low means, what's likely benign vs. worth
   discussing.
3. A **questions for my doctor** list.

The model only sees structured rows (no patient-identifying header text), and
nothing leaves your machine.


In [ ]:
# --- Cell 7: Brief -------------------------------------------------------
import textwrap, requests

def _ollama_chat(prompt: str, model: str | None = None) -> str:
    model = model or OLLAMA_MODEL
    r = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False,
              "options": {"temperature": 0.2}},
        timeout=300,
    )
    return r.json().get("response", "").strip()


def _markers_of_interest() -> list[dict]:
    """Anything flagged on the latest draw OR trending worsening."""
    interesting = []
    seen = set()
    latest = max((r["draw_date"] for r in FLAGGED), default=None)
    for r in FLAGGED:
        if r["unknown"] or r["value_num"] is None:
            continue
        is_flagged_latest = (
            r["draw_date"] == latest
            and r["status"] in {"mild_out", "significant_out"}
        )
        trend_status = next(
            (t["trend"] for t in trend_rows if t["marker"] == r["marker"]),
            None,
        )
        if is_flagged_latest or trend_status == "worsening":
            key = (r["marker"], r["draw_date"])
            if key not in seen:
                seen.add(key)
                interesting.append({**r, "trend": trend_status})
    return interesting


def _ref_for_prompt(marker: str) -> dict:
    ref = by_marker_ref[marker]
    return {
        "canonical_name": ref["canonical_name"],
        "unit": ref["unit"],
        "optimal_range": ref["optimal_range"],
        "clinical_range": ref["clinical_range"],
        "what_it_measures": ref["what_it_measures"],
        "why_it_matters": ref["why_it_matters"],
    }


interesting = _markers_of_interest()
print(f"{len(interesting)} marker-events going into the brief.")

context_payload = {
    "draw_dates": sorted({r["draw_date"] for r in FLAGGED}),
    "markers": [
        {
            "marker": r["marker"],
            "draw_date": r["draw_date"],
            "value": r["value_num"],
            "unit": r.get("unit", ""),
            "status": r["status"],
            "direction": r["direction"],
            "trend": r.get("trend"),
            "reference": _ref_for_prompt(r["marker"]),
        }
        for r in interesting
    ],
    "trends": trend_rows,
}

prompt = textwrap.dedent(f"""
    You are a careful, non-diagnostic health-literacy assistant. You will
    produce a written brief for a layperson based ONLY on the structured
    lab data below. Rules:

      • Do NOT diagnose. Do NOT recommend treatment or medication.
      • Frame everything as "worth discussing with your clinician".
      • Be specific and quantitative -- cite the value and the optimal range.
      • Where a trend is worsening but still in range, say so explicitly.
      • Keep paragraphs short. Use Markdown headings.

    Produce exactly these sections, in order:

      ## Executive Summary
      (3-6 sentences. What's the headline across the dates shown?)

      ## What stood out
      (For EACH marker in the input, a short subsection with: what it
      measures, what its current value and trend suggest in lay terms,
      and what is reasonable to ask about. Use the marker name as a
      level-3 heading.)

      ## Questions for your doctor
      (A bulleted list of 5-10 concrete, specific questions, grounded
      in the markers above.)

      ## What this brief is not
      (One short paragraph reiterating that this is educational, not
      diagnostic, and that reference ranges vary.)

    LAB DATA (JSON):
    {json.dumps(context_payload, indent=2)}
""").strip()

print("\nAsking Ollama for the brief (this can take 30-90s)...")
BRIEF = _ollama_chat(prompt)
print("\n" + "=" * 72)
print(BRIEF[:3000] + ("\n... (truncated in preview)" if len(BRIEF) > 3000 else ""))


## Cell 8 — Export

Write three artifacts to `./output/`:

- `brief.md` — the full Markdown brief Ollama produced.
- `trends.png` — the trend grid from cell 6.
- `flagged_markers.csv` — a flat table of every parsed row with its status.

These three files together are what you'd bring to a doctor's appointment.


In [ ]:
# --- Cell 8: Export ------------------------------------------------------
import pandas as pd
from datetime import datetime

brief_path = OUTPUT_DIR / "brief.md"
csv_path = OUTPUT_DIR / "flagged_markers.csv"

# Brief
header = textwrap.dedent(f"""    # Biomarker Brief
    *Generated locally on {datetime.now().strftime("%Y-%m-%d %H:%M")} using model `{OLLAMA_MODEL}`.*

    > **Not medical advice.** Educational summary only. Bring this to your
    > doctor. Do not act on it.

    """)
brief_path.write_text(header + "\n" + BRIEF + "\n", encoding="utf-8")

# Flagged CSV
export_cols = [
    "draw_date", "marker", "marker_raw", "value", "unit",
    "status", "direction", "category", "source_file",
]
fdf_out = pd.DataFrame(FLAGGED)
for c in export_cols:
    if c not in fdf_out.columns:
        fdf_out[c] = ""
fdf_out = fdf_out[export_cols].sort_values(["draw_date", "marker"])
fdf_out.to_csv(csv_path, index=False)

print("Wrote:")
print(f"  {brief_path}")
print(f"  {OUTPUT_DIR / 'trends.png'}")
print(f"  {csv_path}")
print("\nDone. Open ./output/brief.md to read the briefing.")
